In [18]:
import os
import re
import pandas as pd

gh = pd.read_csv("~/Downloads/pillar_data_with_curation_v4.csv")

# Specify SGE datasets
chromosome_datasets = [
    'BARD1_unpublished', 'CTCF_unpublished', 'NBN_unpublished',
    'PALB2_unpublished', 'XRCC2_unpublished', 'SFPQ_unpublished',
    'RAD51D_unpublished'
]

#some datasets have a c.provided already

cdot_datasets = ['BRCA1_Findlay_2018',"VHL_Buckley_2024","BRCA2_Sahu_2023_exon13_SGE",
                 "BRCA2_Sahu_2023_exon13_Cisplatin","BRCA2_Sahu_2023_exon13_Olaparib","RHO_Wan_2019"]

amino_acid_to_codon = {
    'F': ['TTT', 'TTC'], 'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
    'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'], 'Y': ['TAT', 'TAC'],
    'Ter': ['TAA', 'TAG', 'TGA'], '*': ['TAA', 'TAG', 'TGA'],
    'C': ['TGT', 'TGC'], 'W': ['TGG'], 'P': ['CCT', 'CCC', 'CCA', 'CCG'],
    'H': ['CAT', 'CAC'], 'Q': ['CAA', 'CAG'], 'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'I': ['ATT', 'ATC', 'ATA'], 'M': ['ATG'], 'T': ['ACT', 'ACC', 'ACA', 'ACG'],
    'N': ['AAT', 'AAC'], 'K': ['AAA', 'AAG'], 'V': ['GTT', 'GTC', 'GTA', 'GTG'],
    'A': ['GCT', 'GCC', 'GCA', 'GCG'], 'D': ['GAT', 'GAC'], 'E': ['GAA', 'GAG'],
    'G': ['GGT', 'GGC', 'GGA', 'GGG'],
    '-': ['del']
    
}

output_folder = "./VEP_input/"
os.makedirs(output_folder, exist_ok=True)

# Process datasets
for dataset, group in gh.groupby('Dataset'):
    
    output_file = os.path.join(output_folder, f"{dataset}_vep_input.txt")
    
    with open(output_file, "w") as file:
        # Check if the dataset requires chromosome-based notation
        if dataset in chromosome_datasets:
            for i, (chrom, pos, ref, alt) in enumerate(zip(group['Chrom'], group['hg38_start'], group['ref_allele'], group['alt_allele'])):
                try:
                    chrom = int(chrom)  
                    pos = int(pos) 
                    chrom_str = f"{chrom:02}" if chrom < 10 else str(chrom)
                    notation = f"NC_0000{chrom_str}:g.{pos}{ref}>{alt}"
                    file.write(notation + "\n")
                except KeyError as e:
                    print(f"KeyError for row {i} with value {chrom}: {e}")
                    continue
                except ValueError as e:
                    print(f"ValueError for row {i} with position {pos}: {e}")
                    continue
                    
        elif dataset in cdot_datasets:
            for i, (hgvs_c,ID) in enumerate(zip(group['hgvs_c'],group['Ensembl_transript_ID'])):
                if pd.notna(hgvs_c):
                    transcript = re.sub(r"(ENST\d+)\.\d+", r"\1", ID)
                    file.write(f"{transcript}:{hgvs_c}\n")
        else:
            # Use transcript-based notation for other datasets
            for i, (value, pos, ID, aa_ref) in enumerate(zip(group['aa_alt'], group['aa_pos'], group['Ensembl_transript_ID'], group['aa_ref'])):
                try:
                    amino_acid = aa_ref if value in ['0', '='] else value

                    if pd.notna(ID) and isinstance(ID, str) and amino_acid in amino_acid_to_codon:
                        transcript = re.sub(r"(ENST\d+)\.\d+", r"\1", ID)
                        codons = amino_acid_to_codon[amino_acid]
                        nucleotide_pos = (int(float(pos)) * 3) - 2 
                        for codon in codons:
                            if amino_acid == '-':
                                file.write(f"{transcript}:c.{int(nucleotide_pos)}_{int(nucleotide_pos + 2)}{codon}\n")
                            else:
                                file.write(f"{transcript}:c.{int(nucleotide_pos)}_{int(nucleotide_pos+2)}delins{codon}\n")
                except KeyError as e:
                    print(f"KeyError for row {i} with value {amino_acid}: {e}")
                    continue
                except ValueError as e:
                    print(f"ValueError for row {i} with position {pos}: {e}")
                    continue

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (3,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,33,35) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


ValueError for row 10234 with position nan: cannot convert float NaN to integer
ValueError for row 0 with position CN5: could not convert string to float: 'CN5'
ValueError for row 5805 with position nan: cannot convert float NaN to integer
ValueError for row 3950 with position nan: cannot convert float NaN to integer


In [16]:
#PTEN Mighell needs special input creation, you will have to trash the file created by the code above and replace it with this file 
import pandas as pd
import re

PTEN_mighell = pd.read_excel("~/Downloads/Pillar_project_data_files/PTEN_Mighell_2018.xlsx", header=1)


# Extract numbers from the 'Variant (one letter)' column
PTEN_mighell['aa_pos'] = PTEN_mighell['Variant (one letter)'].astype(str).str.extract(r'(\d+)').astype(float)

file = 'PTEN_Mighell_2018_vep_input.txt'

with open(file, "w") as file:
    for i, (mut_codon,aa_pos) in enumerate(zip(PTEN_mighell['Mut_codon'], PTEN_mighell['aa_pos'])):
        try:
            transcript = "ENST00000371953"
            pos = (int(float(aa_pos)) * 3) - 2
            codon = mut_codon
            key = f"{transcript}:c.{int(pos)}_{int(pos + 2)}delins{codon}"
            file.write(key + "\n")
        except KeyError as e:
            print(f"KeyError for row {i} with value {chrom}: {e}")
            continue
        except ValueError as e:
            print(f"ValueError for row {i} with position {pos}: {e}")
            continue

In [1]:
#concatenate the VEP outputs

import pandas as pd
import glob
import os

folder_path = "/Users/malvikatejura/Downloads/VEP_output" 

file_paths = glob.glob(os.path.join(folder_path, "*.txt"))

dataframes = [pd.read_csv(file, delimiter="\t") for file in file_paths]  

# Concatenate all DataFrames
concatenated_df = pd.concat(dataframes, ignore_index=True)

concatenated_df_2 = concatenated_df.drop_duplicates()

concatenated_df_2.to_csv("VEP_output_v4.csv", index=False)


/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1367/1903555716.py:11: DtypeWarning: Columns (23) have mixed types.Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(file, delimiter="\t") for file in file_paths]
/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1367/1903555716.py:11: DtypeWarning: Columns (16,23) have mixed types.Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(file, delimiter="\t") for file in file_paths]
/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1367/1903555716.py:11: DtypeWarning: Columns (23,30) have mixed types.Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(file, delimiter="\t") for file in file_paths]
/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1367/1903555716.py:11: DtypeWarning: Columns (30) have mixed types.Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(file, delimiter=

In [2]:
#concatenate the gnomad outputs

import pandas as pd
import glob
import os

folder_path = "/Users/malvikatejura/Downloads/Gnomad_v4" 

file_paths = glob.glob(os.path.join(folder_path, "*.csv"))

dataframes = [pd.read_csv(file, delimiter=",") for file in file_paths]  

# Concatenate all DataFrames
concatenated_df = pd.concat(dataframes, ignore_index=True)

concatenated_df_2 = concatenated_df.drop_duplicates()

concatenated_df_2.to_csv("gnomad_v4_01242025.csv", index=False)


/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1367/4074712507.py:11: DtypeWarning: Columns (16) have mixed types.Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(file, delimiter=",") for file in file_paths]
